# Heart Disease Prediction using Machine Learning

**Course:** Fundamentals of Machine Learning – Programming Assignment
**Group members:** *MS26923406*, *MS26923338*
**Dataset:** UCI Heart Disease Dataset (Cleveland subset)
**Dataset source:** https://archive.ics.uci.edu/dataset/45/heart+disease  

**Problem:** Predict whether a patient has heart disease from clinical measurements (binary classification, supervised learning).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, classification_report)

RANDOM_STATE = 42
sns.set_style("whitegrid")

In [ ]:
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach","exang","oldpeak","slope","ca","thal","target"]

def load_data():

    # Try loading the dataset using ucimlrepo
    try:
        from ucimlrepo import fetch_ucirepo
        hd = fetch_ucirepo(id=45)
        df = pd.concat([hd.data.features, hd.data.targets], axis=1)
        df.columns = COLS
        print("Loaded via ucimlrepo")
        return df
    except Exception as e:
        print("ucimlrepo failed:", type(e).__name__)

    # Try loading the dataset directly from the UCI website
    try:
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
        df = pd.read_csv(url, header=None, names=COLS, na_values="?")
        print("Loaded from UCI URL")
        return df
    except Exception as e:
        print("UCI URL failed:", type(e).__name__)

    # If both online methods fail, load the local heart.csv file
    df = pd.read_csv("heart.csv")
    if df.shape[1] != len(COLS):
        raise ValueError("heart.csv must contain the 14 Cleveland dataset columns.")
    df.columns = COLS
    print("Dataset loaded from local heart.csv.")
    return df

df = load_data()

# Convert values to numbers and turn invalid values into missing values.
df = df.apply(pd.to_numeric, errors="coerce")

print("Dataset shape:", df.shape)
display(df.head())

Loaded via ucimlrepo
Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0
